In [1]:
from sklearn.preprocessing import StandardScaler

import numpy as np
from numpy import load
import scipy
import matplotlib.pyplot as plt
from scipy.special import legendre
from numpy.linalg import svd

from time import time
from numpy import sin as sin

import math

In [ ]:
import pandas as pd
scale=StandardScaler()


In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from encoding import superposition,inverse_superposition

## def function


# DATA and function visualization

In [4]:
data = load('../data_prep/disease_SR.npy')


`cheking that can we retrive the graph from super position matrix`

`scaled the 12 lead data before the input inversed_scaled is thae data which is inversed scaled from the superposition matrix `

In [5]:
import numpy as np
import pandas as pd

from encoding import normalize_matrix

# ==============================
# CONFIGURATION
# ==============================

DATASETS = {
    "ST": "../data_prep/unq_disease_ST.npy",
    "SB": "../data_prep/unq_disease_SB.npy",
    "SR": "../data_prep/unq_disease_SR.npy",
}
N_SUPERPOSITION = 5000


# ==============================
# CORE EVALUATION FUNCTION
# ==============================

def evaluate_dataset(name, path):
    print(f"\n{'='*50}")
    print(f"  Processing dataset: {name}")
    print(f"{'='*50}")

    # --- Load ---
    raw = np.load(path, allow_pickle=True)

    # --- Sanitize ---
    print("Sanitizing (checking NaNs)...")
    data_array = np.array([np.asarray(raw[i]) for i in range(len(raw))])
    nan_mask = np.isnan(data_array).any(axis=tuple(range(1, data_array.ndim)))

    valid_indices   = np.where(~nan_mask)[0].tolist()
    removed_indices = np.where( nan_mask)[0].tolist()

    print(f"  Total    : {len(raw)}")
    print(f"  Removed  : {len(removed_indices)}")
    print(f"  Valid    : {len(valid_indices)}")

    # --- Reconstruct & Evaluate ---
    print("Evaluating invertibility...")
    mse_list, mae_list, max_abs_list, rel_list = [], [], [], []

    for i, idx in enumerate(valid_indices):
        if i % max(1, len(valid_indices) // 20) == 0:
            print(f"  {i}/{len(valid_indices)}", end="\r")

        cc = data_array[idx]
        sup = superposition(cc, n=5000)
        inversed_scaled = inverse_superposition(sup)
        # apply same preprocessing as superposition does internally
        cc_normalized = normalize_matrix(cc)
        scaler = StandardScaler()
        cc_scaled = scaler.fit_transform(cc_normalized)  # this is what inverse returns


        # Defensive shape check
        if inversed_scaled.shape != cc_scaled.shape:
            raise ValueError(f"Shape mismatch at index {idx}: "
                            f"{inversed_scaled.shape} vs {cc_scaled.shape}")

        error = inversed_scaled - cc_scaled  # Use scaled version for error calculation

        mse_list.append(np.mean(error**2))
        mae_list.append(np.mean(np.abs(error)))
        max_abs_list.append(np.max(np.abs(error)))

        # Relative error (L2 norm)
        rel_error = np.linalg.norm(error) / (np.linalg.norm(cc_scaled) + 1e-12)
        rel_list.append(rel_error)
    print(f"  {len(valid_indices)}/{len(valid_indices)} ✓            ")

    mse_array = np.array(mse_list)
    mae_array = np.array(mae_list)
    max_array = np.array(max_abs_list)
    rel_array = np.array(rel_list)

    # --- Print Summary ---
    print(f"\n--- Results: {name} ---")
    print(f"  Samples evaluated         : {len(valid_indices)}")
    print(f"  MSE  Mean ± Std           : {mse_array.mean():.6e} ± {mse_array.std():.6e}")
    print(f"  MAE  Mean ± Std           : {mae_array.mean():.6e} ± {mae_array.std():.6e}")
    print(f"  Worst-case Max Abs Error  : {max_array.max():.6e}")
    print(f"  Relative Error Mean ± Std : {rel_array.mean():.6e} ± {rel_array.std():.6e}")
    print(f"  Worst-case Relative Error : {rel_array.max():.6e}")
    print(f"  95th percentile MAE       : {np.percentile(mae_array, 95):.6e}")
    print(f"  99th percentile MAE       : {np.percentile(mae_array, 99):.6e}")

    # --- Save Aggregate CSV ---
    agg_df = pd.DataFrame([{
        "dataset":                   name,
        "total_samples":             len(raw),
        "valid_samples":             len(valid_indices),
        "removed_samples":           len(removed_indices),
        "mse_mean":                  mse_array.mean(),
        "mse_std":                   mse_array.std(),
        "mae_mean":                  mae_array.mean(),
        "mae_std":                   mae_array.std(),
        "max_absolute_error_worst":  max_array.max(),
        "relative_error_mean":       rel_array.mean(),
        "relative_error_std":        rel_array.std(),
        "relative_error_worst":      rel_array.max(),
        "mae_95_percentile":         np.percentile(mae_array, 95),
        "mae_99_percentile":         np.percentile(mae_array, 99),
    }])
    agg_path = f"result_of_TYP1_{name}.csv"
    agg_df.to_csv(agg_path, index=False)
    print(f"  Aggregate results saved to : {agg_path}")

    # --- Save Per-Sample CSV ---
    per_sample_df = pd.DataFrame({
        "sample_index":     valid_indices,
        "mse":              mse_array,
        "mae":              mae_array,
        "max_absolute_error": max_array,
        "relative_error":   rel_array,
    })
    per_path = f"result_of_TYP1_{name}_all_samples.csv"
    per_sample_df.to_csv(per_path, index=False)
    print(f"  Per-sample errors saved to : {per_path}")


# ==============================
# RUN
# ==============================

for name, path in DATASETS.items():
    evaluate_dataset(name, path)

print("\nAll datasets processed.")


  Processing dataset: ST
Sanitizing (checking NaNs)...
  Total    : 3000
  Removed  : 9
  Valid    : 2991
Evaluating invertibility...
  2991/2991 ✓            

--- Results: ST ---
  Samples evaluated         : 2991
  MSE  Mean ± Std           : 4.972410e-05 ± 2.214950e-05
  MAE  Mean ± Std           : 4.092359e-03 ± 1.254645e-03
  Worst-case Max Abs Error  : 1.008824e-01
  Relative Error Mean ± Std : 6.987717e-03 ± 1.555509e-03
  Worst-case Relative Error : 9.669080e-03
  95th percentile MAE       : 5.546469e-03
  99th percentile MAE       : 6.016088e-03
  Aggregate results saved to : result_of_TYP1_ST.csv
  Per-sample errors saved to : result_of_TYP1_ST_all_samples.csv

  Processing dataset: SB
Sanitizing (checking NaNs)...
  Total    : 5000
  Removed  : 14
  Valid    : 4986
Evaluating invertibility...
  4986/4986 ✓            

--- Results: SB ---
  Samples evaluated         : 4986
  MSE  Mean ± Std           : 5.953390e-05 ± 1.120397e-05
  MAE  Mean ± Std           : 4.624652e-03 

In [ ]:
DATASETS = {
    "ST": "../data_prep/unq_disease_ST.npy",
    "SB": "../data_prep/unq_disease_SB.npy",
    "SR": "../data_prep/unq_disease_SR.npy",
}
N_SUPERPOSITION = 5000



for name, path in DATASETS.items():
    print(name)




ST
SB
SR
